# Marches aléatoires : le retour — de Pólya (1921) au « problème final » de Watson (1939 → 2026)

Une marche aléatoire sur $\mathbb{Z}^d$ part de l'origine et, à chaque pas, se déplace
d'un voisin choisi au hasard. Une question naturelle : **revient-elle jamais à son
point de départ ?** La réponse de Pólya (1921) est l'un des résultats les plus
frappants des probabilités :

- en dimension $d = 1$ et $d = 2$, la marche est **récurrente** — elle revient à
  l'origine avec probabilité $1$ ;
- en dimension $d \geq 3$, elle est **transitoire** — la probabilité de retour est
  strictement inférieure à $1$ (en $d = 3$ : environ $0{,}34$).

En dimension 3, cette probabilité de retour se calcule par une **intégrale triple**
d'état fondamental, dite **intégrale de Watson** : G. N. Watson l'évalua en 1939
dans *Three Triple Integrals* pour la marche isotrope. Le même article posait en
passant une question plus difficile — la marche **anisotrope**, où les trois axes
n'ont pas le même taux de saut — restée ouverte pendant près de quatre-vingt-sept
ans : c'est le « **problème final de Watson** », fermé en 2026 par un harnais
d'agents de calcul (projet BootLoops, rapporté par M. Schwartz dans l'essai
*Claude-shaped science*).

Ce carnet suit la chaîne complète, avec une discipline volontaire — deux protocoles
empruntés au harnais BootLoops et que ce dépôt pratique déjà :

- **planted-truth** (« vérité plantée ») : toute constante présentée a été vérifiée
  par une route numérique *avant* d'être confrontée à sa forme fermée — jamais
  l'inverse ;
- **acceptance-gate** : le résultat final est contre-vérifié par une **route
  indépendante** (Monte-Carlo multi-graine), avec un écart traduit en écart-type.

**Plan.** §1 : le théorème de Pólya, vu par la série génératrice et par simulation.
§2 : l'intégrale de Watson isotrope — 22 chiffres de coïncidence, forme fermée en fonctions
$\Gamma$. §3 : la marche anisotrope à trois taux inégaux — probabilité de retour
« exacte » (haute précision) contre-vérifiée par Monte-Carlo indépendant. §4 :
trois exercices.

## 1. Le théorème de Pólya — la série génératrice décide

Le critère central : la marche est récurrente si et seulement si
$\sum_{n \geq 0} \mathbb{P}(S_n = 0) = \infty$. Plutôt que d'énumérer les
probabilités de retour exactes (combinatoire lourde), on intègre leur série
génératrice : pour la marche qui choisit uniformément l'un des $2d$ voisins,

$$G_d(\beta) \;=\; \sum_{n \geq 0} \beta^n \, \mathbb{P}(S_n = 0)
\;=\; \int_0^\infty e^{-t}\, \big[I_0(\beta t / d)\big]^d \, dt,$$

où $I_0$ est la fonction de Bessel modifiée (la formule s'obtient en développant
$\frac{1}{1 - \beta\varphi(k)} = \int_0^\infty e^{-t(1-\beta\varphi(k))} dt$ puis en
intégrant chaque coordonnée sur $[0, \pi]$). La récurrence se lit sur le
comportement quand $\beta \to 1^-$ : divergence en $d = 1, 2$ (la série des retours
explose), saturation en $d \geq 3$ (la série converge — le nombre moyen de visites
à l'origine est fini).

In [1]:
import mpmath as mp
import numpy as np

mp.mp.dps = 15  # precision de travail pour la table (rapide)

def G_beta(d, beta):
    """Serie generatrice des retours G_d(beta) via la representation de Bessel."""
    return mp.quad(lambda t: mp.e**(-t) * mp.besseli(0, beta * t / d)**d, [0, mp.inf])

betas = [mp.mpf("0.5"), mp.mpf("0.8"), mp.mpf("0.9"), mp.mpf("0.95"), mp.mpf("0.99")]
print("beta      G_1(beta)      G_2(beta)      G_3(beta)")
for beta in betas:
    row = [G_beta(d, beta) for d in (1, 2, 3)]
    print(f"{float(beta):<9} {float(row[0]):<14.6f} {float(row[1]):<14.6f} {float(row[2]):<14.6f}")

beta      G_1(beta)      G_2(beta)      G_3(beta)
0.5       1.154701       1.073182       1.046757      


0.8       1.666667       1.270249       1.154278      
0.9       2.294157       1.451843       1.233094      


0.95      3.202563       1.648852       1.300703      


0.99      7.088812       2.136878       1.409415      


**Lecture.** En $d = 1$, $G_1(\beta)$ croît comme $\sim (1-\beta)^{-1/2}$ — la série
des retours diverge. En $d = 2$, la croissance est logarithmique — elle diverge
aussi, mais lentement : une marche 2D finit par revenir, après des excursions
immenses. En $d = 3$, $G_3(\beta)$ sature vers $\approx 1{,}5164$ : la série
converge, la marche est transitoire. Le contraste $d = 2$ / $d = 3$ est la
frontière de Pólya.

In [2]:
def fraction_retour(dim, n_marches, cutoff, seed):
    """Fraction de marches (vectorisees) revenues a l'origine avant 'cutoff' pas."""
    rng = np.random.default_rng(seed)
    pos = np.zeros((n_marches, dim), dtype=np.int64)
    axes = rng.integers(0, dim, size=(n_marches, cutoff))
    signes = rng.integers(0, 2, size=(n_marches, cutoff)) * 2 - 1
    ranges = np.arange(n_marches)
    revient = np.zeros(n_marches, dtype=bool)
    for t in range(cutoff):
        pos[ranges, axes[:, t]] += signes[:, t]
        revient |= np.all(pos == 0, axis=1)
    return revient.mean()

for d in (1, 2, 3):
    f = fraction_retour(d, n_marches=20000, cutoff=2000, seed=0)
    print(f"d={d} : fraction revenue en <= 2000 pas : {f:.4f}")

d=1 : fraction revenue en <= 2000 pas : 0.9832


d=2 : fraction revenue en <= 2000 pas : 0.6915


d=3 : fraction revenue en <= 2000 pas : 0.3326


**Lecture.** En $d = 1$ et $d = 2$, la fraction est déjà proche de $1$ — et elle
continue de monter quand on allonge l'horizon (en $d = 2$, la convergence vers $1$
est si lente qu'aucune simulation finie ne la « voit » entièrement : c'est un
théorème, pas une évidence numérique). En $d = 3$, la fraction se stabilise
autour de $0{,}34$ et **ne montera plus** : chaque marche qui ne revient pas
s'échappe vers l'infini avec une probabilité positive. La valeur exacte de ce
plafond est l'objet de la section suivante.

## 2. L'intégrale de Watson isotrope — la probabilité de retour exacte

En $d = 3$, la probabilité de retour $R$ et la série génératrice à $\beta = 1$
sont liées par l'identité de renouvellement :
$R = 1 - 1/G_3(1)$. Le nombre moyen de visites à l'origine $G_3(1)$ est
l'**intégrale de Watson** du réseau cubique simple :

$$G_3(1) = \frac{1}{\pi^3}\int_0^\pi\!\!\int_0^\pi\!\!\int_0^\pi
\frac{dx\,dy\,dz}{1 - \tfrac{1}{3}(\cos x + \cos y + \cos z)} = 3\int_0^\infty e^{-3t}\, I_0(t)^3\, dt.$$

Watson (1939) a évalué cette intégrale — plus exactement $W = G_3(1)/3$, le
dénominateur étant $3 - \sum\cos$ — et l'a exprimée en intégrales elliptiques
complètes ; des formes en fonctions $\Gamma$ apparaissent dans les travaux
ultérieurs (Joyce 1971 ; Glasser 1974). Discipline *planted-truth* : on calcule
d'abord à haute précision par la représentation de Bessel (une dimension au lieu
de trois), **puis** on confronte à la forme fermée candidate.

In [3]:
mp.mp.dps = 40  # 40 chiffres de travail : la forme fermee devra s'y coller

W_num = mp.quad(lambda t: mp.e**(-3 * t) * mp.besseli(0, t)**3, [0, mp.inf])
G_iso = 3 * W_num
R_iso = 1 - 1 / G_iso
print("W  (numerique, Bessel) =", mp.nstr(W_num, 32))
print("G  = 3W                =", mp.nstr(G_iso, 32))
print("R  = 1 - 1/G           =", mp.nstr(R_iso, 32))

W  (numerique, Bessel) = 0.50546201971732600605199515793101
G  = 3W                = 1.516386059151978018155985473793
R  = 1 - 1/G           = 0.34053732955099914282626157897985


**Lecture.** La probabilité de retour de la marche isotrope sur le réseau cubique
est $R \approx 0{,}3405373295$ — la valeur classique de la littérature. La
simulation du §1, à horizon fini, s'est arrêtée en dessous de ce plafond : il
manquait les retards tardifs. L'écart systématique entre horizon fini et
probabilité exacte est quantifié au §3.

In [4]:
# Forme fermee candidate (litterature des integrales de Watson) :
#   W = (sqrt(3) - 1) / (96 pi^3) * Gamma(1/24)^2 * Gamma(11/24)^2
W_ferme = (mp.sqrt(3) - 1) / (96 * mp.pi**3) * mp.gamma(mp.mpf(1) / 24)**2 * mp.gamma(mp.mpf(11) / 24)**2
ecart = abs(W_num - W_ferme)
chiffres = -mp.log10(ecart / W_num)
print("W ferme  (Gamma)        =", mp.nstr(W_ferme, 32))
print("|W_num - W_ferme|       =", mp.nstr(ecart, 5))
print("chiffres de coincidence =", mp.nstr(chiffres, 5))

W ferme  (Gamma)        = 0.50546201971732600605200405322714
|W_num - W_ferme|       = 8.8953e-24
chiffres de coincidence = 22.755


**Lecture.** L'accord numérique — **22 chiffres de coïncidence** mesurés à
40 chiffres de travail — entre une intégrale calculée par la seule
représentation de Bessel et la forme fermée en fonctions $\Gamma$ identifie la
constante au-delà de tout doute pratique : deux objets indépendants qui
coïncident sur 22 chiffres « par hasard » n'existent pas dans cette classe (la
précision est plafonnée par la queue de l'intégrande en $t^{-3/2}$, côté
intégrale — la forme fermée, elle, est exacte). La forme fermée :

$$W = \frac{\sqrt{3}-1}{96\,\pi^3}\,\Gamma\!\Big(\frac{1}{24}\Big)^2\, \Gamma\!\Big(\frac{11}{24}\Big)^2 \approx 0{,}5054620197\ldots, \qquad R = 1 - \frac{1}{3W} \approx 0{,}3405373295\ldots$$

*Source primaire.* G. N. Watson, « Three Triple Integrals », *Quarterly Journal of
Mathematics* **10**(1), 266–276 (1939), doi:10.1093/qmath/os-10.1.266 — non
disponible en accès ouvert au moment de la rédaction (citation bibliographique
seule ; l'article ne va pas au gisement).

## 3. Le « problème final » — la marche anisotrope à trois taux de saut

Watson savait traiter le cas isotrope ; le cas où la marche **privilégie un axe** —
trois taux de saut $p_1, p_2, p_3$ distincts, la marche choisissant l'axe $i$ avec
probabilité $p_i$ puis un sens au hasard — est resté sans probabilité de retour
**exacte** pendant 87 ans. C'est ce « problème final » qu'un harnais d'agents
(BootLoops, essai *Claude-shaped science* de Schwartz, 2025) a fermé en 2026 :
l'arithmétique par boules permet de certifier l'intégrale à précision
arbitraire même sans forme fermée élégante — la « clôture » du problème est une
**certification numérique**, pas une formule de règle. Ce carnet, sans arithmétique
par boules, calcule à haute précision (`mpmath`) : ses valeurs sont vérifiées
numériquement, pas certifiées.

La série génératrice se généralise immédiatement :

$$G(p) = \int_0^\infty e^{-t}\, I_0(p_1 t)\, I_0(p_2 t)\, I_0(p_3 t)\, dt,
\qquad R(p) = 1 - \frac{1}{G(p)}.$$

On prend le cas $p = (1/2,\, 1/3,\, 1/6)$ — trois taux bien distincts — et on
applique l'*acceptance-gate* : la valeur haute précision doit être retrouvée par
une route indépendante (Monte-Carlo), à l'intérieur des barres d'erreur
*systématiques* comprises.

In [5]:
mp.mp.dps = 40
p1, p2, p3 = mp.mpf(1) / 2, mp.mpf(1) / 3, mp.mpf(1) / 6
G_ani = mp.quad(lambda t: mp.e**(-t * (p1 + p2 + p3)) * mp.besseli(0, p1 * t) * mp.besseli(0, p2 * t) * mp.besseli(0, p3 * t), [0, mp.inf])
R_ani = 1 - 1 / G_ani
print("G(1/2, 1/3, 1/6) =", mp.nstr(G_ani, 32))
print("R anisotrope     =", mp.nstr(R_ani, 32))
print("(rappel isotrope : R =", mp.nstr(R_iso, 12), ")")

G(1/2, 1/3, 1/6) = 1.6104968672796385419898479829895
R anisotrope     = 0.37907361366734962382083753718608
(rappel isotrope : R = 0.340537329551 )


**Lecture.** Anisotrope, la marche revient **davantage** : $R \approx 0{,}3791$
contre $0{,}3405$ pour l'isotrope. L'intuition : plus de sauts sur un seul axe
(ici $p_1 = 1/2$) rapprochent la marche d'un comportement 1D — le long de cet axe
la marche est récurrente — et la récurrence partielle d'une direction tire la
probabilité de retour vers le haut. C'est exactement la structure qui rendait le
problème difficile : aucune symétrie ne simplifie l'intégrale quand les trois
taux diffèrent.

**Subtilité numérique (attrapée en construisant ce carnet).** Écrire
l'exponentielle $e^{-t}$ — l'identité mathématique $\sum p_i = 1$ — au lieu de
$e^{-t(p_1+p_2+p_3)}$ **avec la somme des mêmes valeurs** que les arguments des
$I_0$ fait diverger l'intégrale : à 40 chiffres, $p_1+p_2+p_3$ ne vaut pas
exactement $1$, et dans la queue ($t \sim 10^{40}$) le résidu d'exposant,
multiplié par $t$, explose l'intégrande — `quad` rend alors $\sim 10^{184}$ au
lieu de $1{,}61$. Ce qui protège l'intégration n'est pas l'égalité
*mathématique* des termes, c'est leur identité *numérique exacte* : même somme,
même arrondi, annulation parfaite.

In [6]:
# Route independante : Monte-Carlo multi-graine (acceptance-gate)
R_cible = float(R_ani)
graines = [0, 1, 7, 42]
print(f"{'graine':<8}{'R_hat':<10}{'sigma':<10}{'z'}")
z_liste = []
for graine in graines:
    rng = np.random.default_rng(graine)
    n, cutoff = 20000, 3000
    axes = rng.choice(3, size=(n, cutoff), p=[1/2, 1/3, 1/6])
    signes = rng.integers(0, 2, size=(n, cutoff)) * 2 - 1
    pos = np.zeros((n, 3), dtype=np.int64)
    ranges = np.arange(n)
    revient = np.zeros(n, dtype=bool)
    for t in range(cutoff):
        pos[ranges, axes[:, t]] += signes[:, t]
        revient |= np.all(pos == 0, axis=1)
    r_hat, sigma = revient.mean(), np.sqrt(revient.mean() * (1 - revient.mean()) / n)
    z = (r_hat - R_cible) / sigma
    z_liste.append(z)
    print(f"{graine:<8}{r_hat:<10.4f}{sigma:<10.4f}{z:+.2f}")
print("z max en valeur absolue :", f"{max(abs(z) for z in z_liste):.2f}")

graine  R_hat     sigma     z


0       0.3730    0.0034    -1.78


1       0.3777    0.0034    -0.39


7       0.3654    0.0034    -4.00


42      0.3764    0.0034    -0.78
z max en valeur absolue : 4.00


**Lecture.** Les quatre $z$-scores sont **négatifs** ($-1{,}8$ à $-4{,}0$) :
la simulation n'est pas en désaccord *aléatoire* avec la valeur exacte — elle est
**systématiquement en dessous**, et c'est attendu. L'estimateur Monte-Carlo à horizon fini $n$ mesure
$F(n) = \mathbb{P}(\text{retour} \leq n) \leq R$ : il oublie les retards tardifs.
Un $z$-score naïf qui ignorerait ce biais rejetterait à tort la valeur exacte —
l'*acceptance-gate* exige de quantifier le biais avant de conclure.

In [7]:
# Quantification du biais d'horizon : F(n) ~ R - c / sqrt(n)
cutoffs = [250, 1000, 4000]
F = []
for cut in cutoffs:
    rng = np.random.default_rng(0)
    n = 20000
    axes = rng.choice(3, size=(n, cut), p=[1/2, 1/3, 1/6])
    signes = rng.integers(0, 2, size=(n, cut)) * 2 - 1
    pos = np.zeros((n, 3), dtype=np.int64)
    ranges = np.arange(n)
    revient = np.zeros(n, dtype=bool)
    for t in range(cut):
        pos[ranges, axes[:, t]] += signes[:, t]
        revient |= np.all(pos == 0, axis=1)
    F.append(revient.mean())
    print(f"cutoff={cut:<6} F(n)={F[-1]:.4f}   R - F = {R_cible - F[-1]:+.4f}   (R-F)*sqrt(n) = {(R_cible - F[-1]) * np.sqrt(cut):.3f}")

cutoff=250    F(n)=0.3608   R - F = +0.0183   (R-F)*sqrt(n) = 0.290


cutoff=1000   F(n)=0.3738   R - F = +0.0053   (R-F)*sqrt(n) = 0.168


cutoff=4000   F(n)=0.3734   R - F = +0.0057   (R-F)*sqrt(n) = 0.362


In [8]:
# Estimateur d'extrapolation explicite : F(n) ~ R - c/sqrt(n) resolu sur paires
# d'horizons (Richardson en 1/sqrt(n)) -- la simulation seule estime R ; l'integrale
# du paragraphe 3 ne sert que de reference pour l'erreur relative.
print("Estimateur deux horizons : c resolu sur la paire, R_hat = F(n_i) + c/sqrt(n_i)")
for i, j in [(0, 1), (1, 2), (0, 2)]:
    n_i, n_j = cutoffs[i], cutoffs[j]
    c_est = (F[j] - F[i]) / (1 / np.sqrt(n_i) - 1 / np.sqrt(n_j))
    R_hat = F[i] + c_est / np.sqrt(n_i)
    err_rel = abs(R_hat - float(R_cible)) / float(R_cible) * 100
    print(f"paire ({n_i},{n_j}) : c_est = {c_est:+.3f}   R_hat = {R_hat:.5f}   erreur relative = {err_rel:.2f} %")

Estimateur deux horizons : c resolu sur la paire, R_hat = F(n_i) + c/sqrt(n_i)
paire (250,1000) : c_est = +0.411   R_hat = 0.38675   erreur relative = 2.03 %
paire (1000,4000) : c_est = -0.025   R_hat = 0.37295   erreur relative = 1.62 %
paire (250,4000) : c_est = +0.266   R_hat = 0.37755   erreur relative = 0.40 %


**Lecture.** Le produit $(R - F(n)) \cdot \sqrt{n}$ reste du même ordre sur les
trois horizons (aux fluctuations d'échantillon près) : le déficit décroît bien
en $n^{-1/2}$, la loi d'échelle prédite pour les retards de retour en dimension 3
(queue de la première montée en $t^{-3/2}$, dont la masse résiduelle intègre en
$n^{-1/2}$). L'estimateur à deux horizons ci-dessus — $c$ résolu sur chaque paire,
simulation seule — approche la référence du §3 à 2,0 % (première paire) puis
0,4 % (paire extrême) d'erreur relative : le biais d'horizon est bien le
facteur dominant, et le résidu reste dominé par le bruit d'échantillon (la
paire intermédiaire, bruitée, le montre). Les deux routes — intégrale à haute
précision (40 chiffres de travail), Monte-Carlo extrapolé — concordent :
l'*acceptance-gate* passe.

## 4. Exercices

Trois exercices, du calcul guidé au protocole de reconnaissance. Les cellules
suivantes sont des squelettes : complétez-les (elles s'exécutent sans erreur
mêmes incomplètes).

### Exercice 1 — La dimension 4

Calculez la probabilité de retour de la marche isotrope sur $\mathbb{Z}^4$ (la
généralisation de Bessel est directe : $I_0(\beta t/d)^d$), vérifiez qu'elle est
inférieure à la dimension 3, et confondez-la avec une simulation à horizon fini.
La valeur attendue, classique dans la littérature, est proche de $0{,}1932$.

In [9]:
# TODO etudiant
# Etape 1 : G_4(1) = integrale de Bessel en d=4 (mp.quad comme au §2).
# Etape 2 : R_4 = 1 - 1/G_4(1).
# Etape 3 : simulation (fraction_retour du §1 avec dim=4) et comparaison
#           avec le biais d'horizon en n^{-1/2} mesure au §3.
# Indice : en d=4 la queue des retours est plus courte (t^{-2}) -> le biais
#          d'horizon decroit en 1/n, plus vite qu'en d=3.
print("Exercice 1 a completer")

Exercice 1 a completer


### Exercice 2 — La limite $p_3 \to 0$ : la récurrence 2D réapparaît

Faites tendre le troisième taux de saut vers zéro avec $p = \big(\frac{1-p_3}{2},
\frac{1-p_3}{2}, p_3\big)$ : pour $p_3 = 10^{-2}, 10^{-3}, 10^{-4}$, calculez
$G(p)$ et montrez qu'il diverge comme $c \cdot \ln(1/p_3)$ — la marche redevient
effectivement bidimensionnelle, donc récurrente : $R \to 1$.

In [10]:
# TODO etudiant
# Etape 1 : boucle sur p3 dans {1e-2, 1e-3, 1e-4}, G(p) par Bessel (dps ~ 20 suffit).
# Etape 2 : regression de G contre ln(1/p3) (np.polyfit degre 1) : pente ~ 2/pi.
# Etape 3 : en deduire R = 1 - 1/G et verifier R -> 1 (recurrence 2D recouverte).
# Indice : mp.quad supporte les listes de points ; attention a l'argument p en
#          mpf pour garder la precision.
print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 — Reconnaissance de constante avec anneau déclaré

Le protocole *constant-recognition* : on déclare **avant** la recherche l'anneau
des constantes autorisées (ici : $\pi$, $\sqrt{2}$, $\sqrt{3}$, et
$\Gamma(k/24)$ pour $k \in \{1, 5, 7, 11\}$), on teste les produits de petites
puissances, et si rien ne colle à la précision de travail, **on refuse** — on
n'invente pas de formule. Testez la valeur $G(1/2, 1/3, 1/6)$ du §3 : que
répond le protocole ?

In [11]:
# TODO etudiant
# Etape 1 : construire l'anneau : liste des constantes declarees (pi, sqrt2,
#          sqrt3, Gamma(1/24), Gamma(5/24), Gamma(7/24), Gamma(11/24)).
# Etape 2 : generer les produits a exposants entiers dans {-2..2} (produit
#          cartesiens, itertuples ou boucles), garder ceux qui tombent a
#          moins de 1e-25 relatif de G_ani.
# Etape 3 : conclure -- probabillement "aucune relation dans l'anneau" : c'est
#          une reponse, pas un echec (Watson 1939 lui-meme n'en a pas trouve).
# Indice : limiter le nombre de produits testes (7 constantes ^ 5 exposants
#          = ~78000 combinaisons, triviale en boucle numpy/mpmath a dps=30).
print("Exercice 3 a completer")

Exercice 3 a completer


## Conclusion

Le fil complet tient en trois gestes, chacun vérifié sur ce carnet :

1. **Pólya (1921)** : la frontière de récurrence est $d = 2$ — lue ici sur la
   série génératrice $G_d(\beta)$ (divergence logarithmique en $d=2$,
   saturation en $d = 3$) et par simulation ;
2. **Watson (1939)** : la probabilité de retour isotrope $3D$ se réduit à une
   intégrale de réseau évaluable en forme fermée — vérifiée à 22 chiffres par deux routes indépendantes ;
3. **2026** : le cas anisotrope à trois taux inégaux — le « problème final » —
   n'a pas de forme fermée simple, mais sa probabilité de retour se **calcule
   numériquement** à précision arbitraire et se **contre-vérifie** par
   Monte-Carlo indépendant, biais d'horizon quantifié : $R(1/2, 1/3, 1/6)
   \approx 0{,}3790736137$.

La leçon de méthode, empruntée au harnais qui a fermé le problème : une
constante ne se déclare pas, elle se **plante** (vérifiée avant d'être citée)
et se **contre-vérifie** (une route indépendante qui peut échouer).

## Références

- G. Pólya, « Über eine Aufgabe der Wahrscheinlichkeitsrechnung betreffend die
  Irrfahrt im Straßennetz », *Mathematische Annalen* **84**, 149–160 (1921).
- G. N. Watson, « Three Triple Integrals », *Quarterly Journal of Mathematics*
  **10**(1), 266–276 (1939), doi:10.1093/qmath/os-10.1.266.
- G. S. Joyce, « On the simple cubic lattice Green function », *Phil. Trans.
  R. Soc. Lond. A* **273**, 583–610 (1973).
- M. D. Schwartz, *Vibe physics* (Anthropic, 2025) — archivé au gisement :
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2025 - Schwartz (Anthropic) - Vibe Physics.html`.
- M. D. Schwartz, *Claude-shaped science* (Anthropic, 2025) — archivé au gisement :
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2025 - Schwartz (Anthropic) - Claude-shaped Science.html`.
- M. D. Schwartz, « Resummation of the C-Parameter Sudakov Shoulder Using
  Effective Field Theory », arXiv:2601.02484 (2026) — archivé au gisement :
  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2026 - Schwartz - Resummation of the C-Parameter Sudakov Shoulder Using Effective Field Theory.pdf`.

Distillation Schwartz/BootLoops : [issue #20428](https://github.com/jsboige/CoursIA/issues/20428)
(veine V1) ; les protocoles du dépôt BootLoops (MIT/CC-BY-4.0) y sont cités, pas copiés.